## Agents
- LLMs
    - Groq
- Tools
    - wikipedia
    - search


In [1]:
%pip install langchain | tail -n 1
%pip install langchain-groq | tail -n 1
%pip install langchain-core | tail -n 1
%pip install langchain-community | tail -n 1
%pip install ddgs | tail -n 1
%pip install langgraph | tail -n 1
%pip install wikipedia | tail -n 1

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [15]:
## to load api key
import os
from dotenv import load_dotenv

load_dotenv("../.env")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ")

In [20]:
## to load the llm
from langchain_groq import ChatGroq

# model = "openai/gpt-oss-120b"
model = "llama-3.3-70b-versatile"
# model = "meta-llama/llama-4-scout-17b-16e-instruct"
# model = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model=model,
    temperature=0
)

# to test the llm
response = llm.invoke("What is the capital of France during the Nepolionic era?")
print(response)

content='The capital of France during the Napoleonic era was Paris. Napoleon Bonaparte, who rose to power in 1799, declared himself Emperor of France in 1804 and established his imperial capital in Paris, which remained the capital throughout his reign until his abdication in 1814 and again during his brief return to power in 1815, known as the Hundred Days.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 48, 'total_tokens': 128, 'completion_time': 0.264344172, 'completion_tokens_details': None, 'prompt_time': 0.007224271, 'prompt_tokens_details': None, 'queue_time': 0.111887567, 'total_time': 0.271568443}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d2f93-5f8f-7d00-8a95-08e8f84a000c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 80, 'total_toke

In [17]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.globals import set_debug
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# set_debug(True)

# setting up the duckduckgo query tool
duckduckgo_tool = DuckDuckGoSearchResults()

# setting up wikipedia search tool
wiki_api_wrapper = WikipediaAPIWrapper(
    top_k_results=1, # number of pages to summarize
    doc_content_chars_max=10000 # max charager per page
)

# create wiki tool
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api_wrapper)

In [21]:
from langchain.agents import create_agent

tools = [duckduckgo_tool, wiki_tool]

system_prompt = (
    "You are a helpful research assistant. "
    "CRITICAL: You only have access to the 'duckduckgo_tool' and 'wiki_tool' tools. "
    "Do not attempt to use any other tools like 'open_file' or 'browser'. "
    "If you need to visit a website, use 'duckduckgo_search' to find the information."
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    debug=True
)

In [22]:
query = """
why nvidia is not seeing the GPU development by AMD, apple and intel as a threat, however the 
A15 and A16 chips from Tesla as a serious threat?
"""

response = agent.invoke({
    "messages": [
        ("user", query)
    ]
})
print(response["messages"][-1].content)

[values] {'messages': [HumanMessage(content='\nwhy nvidia is not seeing the GPU development by AMD, apple and intel as a threat, however the \nA15 and A16 chips from Tesla as a serious threat?\n', additional_kwargs={}, response_metadata={}, id='0244f3c7-7453-4604-8cbe-9179f4a2a5a7')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vtvffq5t8', 'function': {'arguments': '{"query":"Nvidia views on AMD, Apple, Intel, and Tesla GPU development"}', 'name': 'duckduckgo_results_json'}, 'type': 'function'}, {'id': '8gzx8rgf6', 'function': {'arguments': '{"query":"Nvidia"}', 'name': 'wikipedia'}, 'type': 'function'}, {'id': '03hxa6j25', 'function': {'arguments': '{"query":"Tesla A15 and A16 chips"}', 'name': 'wikipedia'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 487, 'total_tokens': 551, 'completion_time': 0.214464201, 'completion_tokens_details': None, 'prompt_time': 0.080849955, 'promp

In [23]:
from pprint import pprint
pprint(response["messages"][-1].content)

('Nvidia is not seeing the GPU development by AMD, Apple, and Intel as a '
 'threat because they have a strong market share and a wide range of products '
 'that cater to different markets. However, Nvidia views the A15 and A16 chips '
 'from Tesla as a serious threat because they are specifically designed for AI '
 'and deep learning applications, which is a key area of focus for Nvidia. '
 "Tesla's chips are also designed to be highly efficient and scalable, which "
 "could potentially disrupt Nvidia's dominance in the AI and deep learning "
 "market. Additionally, Tesla's partnership with AMD to develop AI chips could "
 "also be seen as a threat to Nvidia's market share.")
